# Quantitative Trading With Machine Learning
## Modular Colab Demonstration

This notebook demonstrates the modular implementation of the VWAGY quantitative-trading project. It downloads adjusted Yahoo Finance data, builds VWAGY and supply-chain features, evaluates four regression models across multiple forecast horizons, selects a final configuration using validation data only, and evaluates a simple transaction-cost-aware trading strategy against VWAGY Buy and Hold.

The original methodology is preserved: chronological 60/20/20 splitting, training-only scaling for Elastic Net, validation-only model and threshold selection, 28-day proxy-return lags, and a 0.1% transaction cost.

## 1. Install and import

Run this notebook in Google Colab. The next cell installs the scientific Python dependencies.

In [ ]:
!pip -q install numpy pandas scikit-learn xgboost lightgbm yfinance matplotlib seaborn

In [ ]:
# If using the GitHub repository, clone the branch containing the modular refactor.
# Change the branch name if you used a different name.
!git clone -b modular-version https://github.com/Mayur-Shashidhar/Quantitative-Trading-With-Machine-Learning.git /content/quantitative-trading-ml
%cd /content/quantitative-trading-ml
!pip -q install -e .

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from quant_trading.config import ProjectConfig
from quant_trading.pipeline import run_experiment

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
print("Imports successful.")

## 2. Configure and run the experiment

The default configuration matches the original project. Yahoo Finance data is downloaded at runtime, so exact benchmark values can change if the provider revises historical data.

In [ ]:
config = ProjectConfig(
    ticker="VWAGY",
    start_date="2015-01-01",
    end_date="2025-12-31",
    train_fraction=0.60,
    validation_fraction=0.20,
    horizons=(1, 5, 10, 15, 20),
    supply_lag_days=28,
    transaction_cost=0.001,
)

result = run_experiment(config)
print("Experiment completed.")
print("Selected model:", result["selected_model"])
print("Selected horizon:", result["selected_horizon"], "day(s)")

## 3. Baseline model comparison

The baseline uses only VWAGY-derived features. Lower RMSE and MAE are better. R² near zero or below zero indicates weak explanatory power.

In [ ]:
baseline_results = result["baseline_validation"].rename(columns={"RMSE": "Validation RMSE", "MAE": "Validation MAE", "R2": "Validation R2"})
display(baseline_results.style.format({"Validation RMSE": "{:.6f}", "Validation MAE": "{:.6f}", "Validation R2": "{:.6f}"}))

## 4. Enhanced model and forecast-horizon comparison

The enhanced experiment adds 28 lagged daily returns for each available supply-chain proxy. The final model and horizon are selected using the lowest validation RMSE.

In [ ]:
horizon_results = result["horizon_results"].copy()
display(horizon_results.sort_values(["Horizon", "RMSE"]).style.format({"RMSE": "{:.6f}", "MAE": "{:.6f}", "R2": "{:.6f}"}))

plt.figure(figsize=(11, 5))
sns.lineplot(data=horizon_results, x="Horizon", y="RMSE", hue="Model", marker="o")
plt.title("Validation RMSE by Forecast Horizon")
plt.xlabel("Forecast horizon in trading days")
plt.ylabel("Validation RMSE")
plt.show()

## 5. Final out-of-sample regression performance

After selecting the model and horizon, the selected model is retrained on the combined training and validation data. The test period is then evaluated once.

In [ ]:
final_regression = pd.DataFrame([result["final_regression"]], index=["Final test"])
display(final_regression.style.format("{:.6f}"))
print("Selected model:", result["selected_model"])
print("Selected horizon:", result["selected_horizon"], "day(s)")

## 6. Trading threshold and position distribution

For the selected one-day configuration, a positive prediction above the threshold creates a long position, a negative prediction below the negative threshold creates a short position, and all other predictions create a neutral position. The threshold is selected using validation data only.

In [ ]:
trading = result["trading"]
if trading is None:
    print("Trading is not activated because the selected horizon is not one day.")
else:
    print(f"Selected threshold: {trading['threshold']:.4f}")
    threshold_table = trading["threshold_results"].copy()
    display(threshold_table.style.format({"Total Return": "{:.4f}", "Annualized Return": "{:.4f}", "Volatility": "{:.4f}", "Sharpe": "{:.4f}"}))
    signals = trading["backtest"]["signals"]
    signal_counts = pd.Series(signals).map({1: "Long", -1: "Short", 0: "Neutral"}).value_counts()
    display(signal_counts.to_frame("Days"))
    signal_counts.plot(kind="bar", color=["#2E8B57", "#B22222", "#808080"], figsize=(7, 4), title="Final Test Position Distribution")
    plt.ylabel("Number of days")
    plt.show()

## 7. ML strategy versus Buy and Hold

The strategy includes the configured 0.1% transaction cost whenever the position changes. Buy and Hold uses the same final test dates.

In [ ]:
if trading is not None:
    comparison = pd.DataFrame({
        "ML Strategy": trading["strategy_metrics"],
        "VWAGY Buy and Hold": trading["buy_hold_metrics"],
    }).T
    display(comparison.style.format("{:.4f}"))

    ml_equity = np.cumprod(1 + trading["backtest"]["net_returns"])
    bh_equity = np.cumprod(1 + trading["buy_hold_returns"])
    plt.figure(figsize=(11, 5))
    plt.plot(ml_equity, label="ML strategy", linewidth=2)
    plt.plot(bh_equity, label="VWAGY Buy and Hold", linewidth=2)
    plt.axhline(1.0, color="black", linestyle="--", linewidth=1, label="Initial value")
    plt.title("ML Strategy Equity Curve on Final Test Period")
    plt.xlabel("Test-period trading days")
    plt.ylabel("Portfolio value")
    plt.legend()
    plt.show()

## 8. Interpretation

A negative or near-zero R² means next-day return prediction is difficult for this feature set. If the strategy loses money while Buy and Hold performs better, the correct conclusion is not that the entire field of machine-learning trading is impossible. The defensible conclusion is that this particular data, proxy universe, model set, threshold rule, and test period did not produce sufficient out-of-sample signal.

The result is still valuable because it demonstrates a leakage-aware research process and shows why transaction costs, chronological validation, and a simple benchmark are essential.

## 9. Viva talking points

- The project is a regression problem because the target is a continuous future return.
- The chronological split prevents future regimes from entering training.
- Elastic Net is scaled using training data only because regularization depends on feature scale.
- The final test set is not used for model, horizon, or threshold selection.
- Supply-chain features are proxies, not proof of causal supplier relationships.
- The modular refactor separates data, features, models, evaluation, strategy, and orchestration.
- The result is an empirical research pipeline, not investment advice or a live trading system.

## 10. Limitations and future work

The current project does not model slippage, bid-ask spread, liquidity, short-borrow fees, or capital allocation. Yahoo Finance data can change, the proxy universe is limited, and only one final test period is used. Stronger future work would use walk-forward validation across multiple market regimes, realistic execution costs, broader macroeconomic data, multiple assets, statistical significance tests, and stability analysis of selected features.